# 82514 · Sesión S18 — Localización con EKF en el almacén

**Bloque 6** · jueves 19 de noviembre de 2026 · 2 h · IQS Universitat Ramon Llull

El EKF del almacén: error, elipse y cuándo falla.

| | |
|---|---|
| **En clase** | Al acabar el S18a, abrid este y ejecutad la primera celda. En la diapositiva 64, ocho minutos: secciones 1 y 2 sin deteneros; de la 3 a la 7, ejecutad y leed; en la 6, añadid distancias; y «✎ Vuestra respuesta». |
| **En casa** | Sección 8 y ejercicios; soluciones al final. |
| **Entrega** | Al acabar la clase, individual: este cuaderno y el S18a, ejecutados y con «✎ Vuestra respuesta» escrita (Archivo → Descargar → Descargar .ipynb), a la Tarea «Taller S18: estimación y localización» de Moodle. Sin nota: presentado o no presentado. |

Cada paso tiene tres celdas: **▶ Parámetros**, lo único que se cambia; **⚙ Cálculo**, que no hace falta tocar; y **✔ Resultado**, lo que sale.

---

In [ ]:
# @title ⚙ Instalación · ejecutad esta celda primero
# Ejecutar una sola vez. En Colab tarda unos segundos.
import importlib, subprocess, sys

def asegurar(mods):
    faltan = []
    for pip_name, import_name in mods:
        try:
            importlib.import_module(import_name)
        except ImportError:
            faltan.append(pip_name)
    if faltan:
        print('Instalando:', ' '.join(faltan))
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + faltan, check=False)
    else:
        print('Todo instalado ya.')

asegurar([('numpy', 'numpy'), ('matplotlib', 'matplotlib')])

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

VERSIONES = {'numpy': '2.4.6', 'matplotlib': '3.10.0'}   # las de todo el bloque 6 (las de Colab al prepararlo)
EN_COLAB = 'google.colab' in sys.modules
np.set_printoptions(precision=3, suppress=True)
plt.rcParams['figure.figsize'] = (10, 3.4)
plt.rcParams['axes.grid'] = True
IQS_AZUL, IQS_VERDE, GRIS, AMBAR, ROJO = '#1B2A80', '#1FA355', '#8A8FA8', '#B9770E', '#C0392B'
COMA = FuncFormatter(lambda v, p: f'{v:g}'.replace('.', ',').replace('-', '−'))   # ejes con coma decimal

def coma(x, n=2):
    """Un número con coma decimal y signo menos, para imprimir."""
    return f'{x:.{n}f}'.replace('.', ',').replace('-', '−')

def cien(v, n=2):
    """Un número pequeño o grande con coma decimal y potencia de diez: 1,40·10⁻¹⁰."""
    m, e = f'{v:.{n}e}'.split('e')
    return m.replace('.', ',').replace('-', '−') + '·10' + str(int(e)).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))

def ejes(*axs):
    for ax in axs:
        ax.xaxis.set_major_formatter(COMA); ax.yaxis.set_major_formatter(COMA)

print('numpy', np.__version__, '· matplotlib', matplotlib.__version__, '· en Colab:', EN_COLAB)
if not np.__version__.startswith('2.4'):
    print('AVISO: este cuaderno se probó con numpy', VERSIONES['numpy'], '; las cifras con semilla pueden cambiar un poco.')

## 1. El almacén, el robot y las balizas

*En clase, sin deteneros · diapositivas 46, 61 y 62*

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# el ruido de la odometría (Q): σ del avance tras un metro [m] (13,4 mm en cada paso de 5 cm)
ODO_POR_METRO = 0.06  # @param {type:"number"}
# σ del giro en cada paso de 5 cm [°] (0,9° por metro)
GIRO_POR_PASO = 0.2  # @param {type:"number"}
# el ruido del sensor de balizas (R): σ de la distancia [m] y del rumbo [°]
SIGMA_DISTANCIA = 0.05  # @param {type:"number"}
SIGMA_RUMBO = 1.0  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo · el simulador: ejecutadlo sin abrirlo
# El almacen del curso y el robot movil del bloque 6 (S18 y S19). Es la celda del simulador del cuaderno S18b:
# las cifras de las diapositivas salen de este mismo codigo.
#  - El almacen de S22: rejilla de 90 x 130 celdas de 10 cm (13 x 9 m), dos naves, un muro con una puerta
#    ancha (2,0 m) y una estrecha (0,8 m), y estanterias. El muelle en (0,6; 0,6) y el puesto de picking en (8,3; 8,2).
#  - El robot: diferencial, con la rueda de S6; a 0,5 m/s, con la odometria a 10 Hz (pasos de 5 cm).
#    Ruido de la odometria: 6 cm y 0,9° por metro recorrido (por paso, 13,4 mm y 0,2°), en ▶ Parametros.
#  - Diez balizas reflectantes y un sensor de distancia y rumbo: 5 cm y 1° (en ▶ Parametros), alcance de 8 m con
#    linea de vista, un barrido por segundo (cada 10 pasos de odometria, 0,5 m).
import numpy as np

D2R = np.pi / 180
RES, FILAS, COLS = 0.10, 90, 130            # celdas de 10 cm: 13 m en x, 9 m en y


def mapa_almacen():
    """La rejilla de ocupacion del almacen de S22 (True = ocupado). Fila = y/0,1; columna = x/0,1."""
    occ = np.zeros((FILAS, COLS), bool)
    occ[0, :] = occ[-1, :] = True              # paredes exteriores
    occ[:, 0] = occ[:, -1] = True
    occ[42:47, :] = True                       # el muro central, de y = 4,2 a 4,7 m
    occ[42:47, 8:28] = False                   #   la puerta ancha, de x = 0,8 a 2,8 m
    occ[42:47, 92:100] = False                 #   la puerta estrecha, de x = 9,2 a 10,0 m
    occ[12:36, 40:47] = True                   # estanterias de la nave de abajo
    occ[12:36, 64:71] = True
    occ[58:78, 34:41] = True                   # estanterias de la nave de arriba
    occ[58:78, 58:65] = True
    occ[58:78, 82:89] = True
    return occ


OCC = mapa_almacen()
MUELLE, PICKING = np.array([0.6, 0.6]), np.array([8.3, 8.2])
BALIZAS = np.array([[0.1, 2.0], [3.0, 0.1], [8.0, 0.1], [12.9, 2.0], [5.5, 4.15],      # nave de abajo
                    [0.1, 7.0], [4.7, 8.9], [10.5, 8.9], [12.9, 6.5], [7.0, 4.75]])    # nave de arriba
RUTA = np.array([[0.6, 0.6], [1.8, 0.6], [1.8, 5.2], [9.6, 5.2], [9.6, 0.6], [0.6, 0.6]])   # la ruta del inventario

V, HZ = 0.5, 10                              # 0,5 m/s; odometria a 10 Hz
DD = V / HZ                                  # 5 cm por paso
SIG_D, SIG_T = ODO_POR_METRO / np.sqrt(20), GIRO_POR_PASO * D2R   # por paso: 13,4 mm y 0,2° (6 cm y 0,9° por metro)
Q = np.diag([SIG_D**2, SIG_T**2])            # Q: la covarianza del ruido de proceso (la odometria)
SIG_R, SIG_PHI, ALCANCE = SIGMA_DISTANCIA, SIGMA_RUMBO * D2R, 8.0
R = np.diag([SIG_R**2, SIG_PHI**2])          # R: la covarianza del ruido de medida (el sensor de balizas)
CADA = 10                                    # un barrido de balizas cada 10 pasos: una vez por segundo
K95 = 2.4477                                 # en el plano, la elipse de 2,45σ encierra el 95 %


def envuelve(a):
    """Un angulo en (−π, π]: sin esto, la innovacion del rumbo salta 360°."""
    return (a + np.pi) % (2 * np.pi) - np.pi


def paso(x, d):
    """El modelo de movimiento: la pose x = (x; y; θ) compuesta con la odometria d = (δ_d; δ_θ) en SE(2),
    como en S8: avanzar δ_d en la direccion del rumbo y despues girar δ_θ."""
    return np.array([x[0] + d[0] * np.cos(x[2]), x[1] + d[0] * np.sin(x[2]), envuelve(x[2] + d[1])])


def jac_paso(x, d):
    """Sus jacobianos: F_x respecto a la pose (3 x 3) y F_v respecto a la odometria (3 x 2)."""
    c, s = np.cos(x[2]), np.sin(x[2])
    Fx = np.array([[1.0, 0.0, -d[0] * s], [0.0, 1.0, d[0] * c], [0.0, 0.0, 1.0]])
    Fv = np.array([[c, 0.0], [s, 0.0], [0.0, 1.0]])
    return Fx, Fv


def h(x, b):
    """El modelo de medida: distancia y rumbo a la baliza b desde la pose x."""
    dx, dy = b[0] - x[0], b[1] - x[1]
    return np.array([np.hypot(dx, dy), envuelve(np.arctan2(dy, dx) - x[2])])


def jac_h(x, b):
    """Su jacobiano H_x respecto a la pose (2 x 3)."""
    dx, dy = b[0] - x[0], b[1] - x[1]
    r2 = dx * dx + dy * dy
    r = np.sqrt(r2)
    return np.array([[-dx / r, -dy / r, 0.0], [dy / r2, -dx / r2, -1.0]])


def libre_vista(a, b):
    """¿Ve el sensor la baliza b desde a? Sin celdas ocupadas en medio (la baliza esta en la pared)."""
    n = int(np.ceil(np.linalg.norm(b - a) / 0.02))
    p = a + np.linspace(0, 1, n)[:, None] * (b - a)
    cc = np.clip((p[:, 0] / RES).astype(int), 0, COLS - 1)
    ff = np.clip((p[:, 1] / RES).astype(int), 0, FILAS - 1)
    return not OCC[ff, cc][:-6].any()


def ruta_densa():
    """La ruta del inventario, en pasos de 5 cm."""
    pts = [RUTA[0]]
    for a, b in zip(RUTA[:-1], RUTA[1:]):
        n = int(round(np.linalg.norm(b - a) / DD))
        pts += [a + (b - a) * k / n for k in range(1, n + 1)]
    return np.array(pts)


def mandos(rd):
    """La odometria verdadera de cada paso: el avance y el giro que llevan de un punto de la ruta al siguiente."""
    d, th = [], 0.0
    for a, b in zip(rd[:-1], rd[1:]):
        thn = np.arctan2(b[1] - a[1], b[0] - a[0])
        d.append((np.linalg.norm(b - a), envuelve(thn - th)))
        th = thn
    return np.array(d)


RD = ruta_densa()
U = mandos(RD)                               # 544 pasos, 27,2 m
VERDAD = [np.array([RD[0, 0], RD[0, 1], 0.0])]
for d in U:
    VERDAD.append(paso(VERDAD[-1], d))
VERDAD = np.array(VERDAD)
VISIBLES = [[j for j, b in enumerate(BALIZAS) if np.linalg.norm(b - p[:2]) < ALCANCE and libre_vista(p[:2], b)]
            for p in VERDAD]


def simula(semilla=1):
    """Lo que leen los sensores en una pasada por la ruta: la odometria de cada paso y, una vez por segundo,
    la distancia y el rumbo de cada baliza visible (con su numero: la asociacion se supone conocida)."""
    rng = np.random.default_rng(semilla)
    odo = U + rng.normal(0, [SIG_D, SIG_T], size=U.shape)
    med = {}
    for k in range(CADA, len(VERDAD), CADA):
        med[k] = [(j, h(VERDAD[k], BALIZAS[j]) + rng.normal(0, [SIG_R, SIG_PHI])) for j in VISIBLES[k]]
    return odo, med


SIGMA0 = np.diag([0.02**2, 0.02**2, (1.0 * D2R)**2])   # al salir del muelle: 2 cm y 1°


def ekf(odo, med, balizas=range(10), corregir=True, x0=None, Sigma0=SIGMA0, cambia=None, puerta=None):
    """El EKF de localizacion: predice con la odometria en cada paso y corrige con cada baliza vista.
    balizas: las que estan en el mapa. corregir=False: a estima, solo prediccion.
    cambia = (k, j_real, j_usada): en el barrido k, la medida de la baliza j_real se atribuye a j_usada.
    puerta: si se da (por ejemplo 9,21), se descartan las medidas con d² = νᵀ·S⁻¹·ν por encima.
    Devuelve las poses, las covarianzas, las d² de cada medida y los saltos de la correccion."""
    x = VERDAD[0].copy() if x0 is None else np.array(x0, float)
    S_ = Sigma0.copy()
    X, P, D2, SALTOS = [x.copy()], [S_.copy()], [], []
    for k, d in enumerate(odo, start=1):
        Fx, Fv = jac_paso(x, d)                       # prediccion
        x = paso(x, d)
        S_ = Fx @ S_ @ Fx.T + Fv @ Q @ Fv.T
        if corregir and k in med:                     # correccion, una baliza cada vez
            x_antes = x.copy()
            for j, z in med[k]:
                if j not in balizas:
                    continue
                jm = cambia[2] if (cambia and k == cambia[0] and j == cambia[1]) else j
                Hx = jac_h(x, BALIZAS[jm])
                nu = z - h(x, BALIZAS[jm])
                nu[1] = envuelve(nu[1])               # la innovacion
                Sm = Hx @ S_ @ Hx.T + R                # su covarianza
                d2 = float(nu @ np.linalg.solve(Sm, nu))   # Mahalanobis
                D2.append((k, j, jm, d2))
                if puerta is not None and d2 > puerta:
                    continue
                L = S_ @ Hx.T @ np.linalg.inv(Sm)      # la ganancia de Kalman
                x = x + L @ nu
                x[2] = envuelve(x[2])
                S_ = (np.eye(3) - L @ Hx) @ S_
            SALTOS.append((k, np.linalg.norm(x[:2] - x_antes[:2])))
        X.append(x.copy()); P.append(S_.copy())
    return np.array(X), np.array(P), D2, SALTOS


def metricas(X, P):
    """Error de posicion frente a la verdad: RMSE, maximo y final; y la consistencia: el % de pasos con el
    error dentro de la elipse del 95 % (2,45σ) de su propia Σ."""
    e = X[:, :2] - VERDAD[:, :2]
    n = np.linalg.norm(e, axis=1)
    dentro = [ei @ np.linalg.solve(Pi[:2, :2], ei) <= K95**2 for ei, Pi in zip(e[1:], P[1:])]
    return dict(rmse=np.sqrt((n**2).mean()), maximo=n.max(), final=n[-1], dentro=100 * np.mean(dentro), err=n)


def elipse(Sig, n=K95, m=60):
    """Los puntos de la elipse de n·σ de una covarianza 2 x 2 (sus semiejes, n·√ de los autovalores)."""
    w, v = np.linalg.eigh(Sig)
    t = np.linspace(0, 2 * np.pi, m)
    return (v @ (n * np.sqrt(np.maximum(w, 0))[:, None] * np.array([np.cos(t), np.sin(t)]))).T

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
L_ruta = np.sum(np.linalg.norm(np.diff(RUTA, axis=0), axis=1))   # la longitud de la ruta [m]
nv = [len(v) for v in VISIBLES]                                   # cuántas balizas ve en cada paso
T_PASOS = np.arange(len(VERDAD)) / HZ                             # el tiempo de cada paso [s]

def dibuja_mapa(ax, en_mapa=range(10), numeros=False):
    """Para dibujar: el almacén (lo ocupado, en gris), las balizas del mapa (estrellas verdes; huecas, las que no
    están en el mapa), con su número si numeros, y el muelle y el puesto de picking (en ámbar)."""
    ax.imshow(OCC, origin='lower', extent=(0, 13, 0, 9), cmap='Greys', vmin=0, vmax=1.6, interpolation='nearest')
    si = [j for j in range(len(BALIZAS)) if j in en_mapa]
    no = [j for j in range(len(BALIZAS)) if j not in en_mapa]
    ax.plot(BALIZAS[si, 0], BALIZAS[si, 1], '*', color=IQS_VERDE, ms=13, mec='white', mew=0.6, zorder=6)
    if no:
        ax.plot(BALIZAS[no, 0], BALIZAS[no, 1], '*', mfc='white', mec=GRIS, ms=13, mew=1.2, zorder=6)
    if numeros:               # el número, fuera de la pared; las del muro, debajo (la 4) o encima de la ruta (la 9)
        for j, (bx, by) in enumerate(BALIZAS):
            xy, al = (((bx - 0.3, by), dict(ha='right')) if bx < 1 else ((bx + 0.3, by), dict(ha='left')) if bx > 12 else
                      ((bx, by - 0.25), dict(va='top')) if by < 1 else ((bx, by + 0.25), dict(va='bottom')) if by > 8 else
                      ((bx, 3.9), dict(va='top')) if by < 4.45 else ((bx, 5.4), dict(va='bottom')))
            ax.text(*xy, str(j), fontsize=10, fontweight='bold', color=IQS_VERDE if j in en_mapa else GRIS, zorder=7,
                    **{'ha': 'center', 'va': 'center', **al}, bbox=dict(boxstyle='round,pad=0.1', fc='white', ec='none'))
    ax.plot(*MUELLE, 's', color=AMBAR, ms=8, zorder=6); ax.plot(*PICKING, 's', color=AMBAR, ms=8, zorder=6)
    ax.text(MUELLE[0] + 0.25, MUELLE[1] - 0.42, 'muelle', fontsize=9, color=AMBAR, zorder=8,     # debajo de la ruta
            bbox=dict(boxstyle='round,pad=0.12', fc='white', ec='none', alpha=0.85))
    ax.text(PICKING[0] + 0.1, PICKING[1] + 0.3, 'picking', fontsize=9, color=AMBAR, zorder=6)
    lim = 0.7 if numeros else 0.4                 # sin números, 0,4 m: las estrellas de las paredes, enteras
    ax.set_xlim(-lim, 13 + lim); ax.set_ylim(-lim, 9 + lim); ax.set_aspect('equal'); ax.grid(False)
    ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)'); ejes(ax)

In [ ]:
# @title ✔ Resultado
print(f'La ruta del inventario: {coma(L_ruta,1)} m en {len(U)} pasos de 5 cm ; balizas a la vista: de {min(nv)} a {max(nv)}')
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.6), gridspec_kw={'width_ratios': [1.45, 1]})
dibuja_mapa(a1, numeros=True)
a1.plot(VERDAD[:, 0], VERDAD[:, 1], color='black', lw=2, label='la ruta (la verdad)')
a1.annotate('', xy=VERDAD[150, :2], xytext=VERDAD[120, :2], arrowprops=dict(arrowstyle='-|>', color='black', lw=2, mutation_scale=18))
a1.plot([], [], '*', color=IQS_VERDE, ms=12, label='balizas, con su número')
a1.legend(fontsize=9, loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, frameon=False)
a2.step(T_PASOS, nv, where='post', color=IQS_VERDE, lw=1.8)
a2.set_ylim(0, 6); a2.set_yticks(range(7)); a2.set_xlabel('t (s)'); a2.set_ylabel('balizas a la vista')
a2.set_title('Cuántas ve en cada paso: hasta 8 m y sin nada en medio', fontsize=11); ejes(a2)
plt.tight_layout(); plt.show()

**Qué tiene que salir:** 27,2 m en 544 pasos; ve de 2 a 5 balizas.

## 2. Los modelos y sus jacobianos, comprobados

*En clase, sin deteneros · diapositivas 60 a 62*

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
def numerico(f, x, h=1e-6):
    """Jacobiano por diferencias centradas, columna a columna (con el ángulo envuelto)."""
    cols = []
    for j in range(len(x)):
        e = np.zeros(len(x)); e[j] = h
        d = f(x + e) - f(x - e)
        if len(d) == 3: d[2] = envuelve(d[2])
        if len(d) == 2: d[1] = envuelve(d[1])
        cols.append(d / (2 * h))
    return np.array(cols).T

xp, dp, b = np.array([1.8, 2.0, 0.6]), np.array([0.05, 0.004]), BALIZAS[9]   # una pose, una odometría y la baliza 9
Fx, Fv = jac_paso(xp, dp)
DIF_FX = np.abs(numerico(lambda x: paso(x, dp), xp) - Fx).max()              # lo que se aparta el numérico
DIF_FV = np.abs(numerico(lambda d: paso(xp, d), dp) - Fv).max()
xr = np.array([6.0, 5.2, 0.0])                                               # el robot en el pasillo, en x = 6,0 m
DIF_HX = np.abs(numerico(lambda x: h(x, b), xr) - jac_h(xr, b)).max()
z = h(xr, b)                                                                 # la distancia [m] y el rumbo [rad]
HX = jac_h(xr, b)
# para dibujar: la distancia y el rumbo si el robot se mueve de lado (en y), de verdad y con la tangente
LADO = np.linspace(-1.0, 1.0, 81)                                            # cuánto se mueve de lado [m]
Z_LADO = np.array([h(xr + np.array([0.0, dy, 0.0]), b) for dy in LADO])      # con h, de verdad
Z_TANGENTE = z + np.outer(LADO, HX[:, 1])                                    # con la columna y de H_x

In [ ]:
# @title ✔ Resultado
print('F_x, diferencia máxima:', cien(DIF_FX))
print('F_v, diferencia máxima:', cien(DIF_FV))
print('H_x, diferencia máxima:', cien(DIF_HX))
print(f'Desde (6,0; 5,2; 0°), la baliza (7,0; 4,75): r = {coma(z[0])} m, φ = {coma(np.degrees(z[1]),1)}°')
print('H_x =', ' ; '.join('(' + ', '.join(coma(v, 3) for v in f) + ')' for f in HX))
fig, axs = plt.subplots(1, 2, figsize=(12, 3.6))
for ax, i, k, t, u in ((axs[0], 0, 1, 'La distancia a la baliza 9', 'r (m)'), (axs[1], 1, 180 / np.pi, 'El rumbo a la baliza 9', 'φ (°)')):
    ax.plot(LADO, Z_LADO[:, i] * k, color=IQS_AZUL, lw=2.4, label='h(x), de verdad')
    ax.plot(LADO, Z_TANGENTE[:, i] * k, '--', color=ROJO, lw=2, label='su tangente, con H_x')
    ax.plot(0, z[i] * k, 'o', color='black', ms=7, zorder=5)
    ax.set_xlabel('el robot, movido de lado desde y = 5,2 m (m)'); ax.set_ylabel(u); ax.set_title(t, fontsize=11)
    ax.legend(fontsize=9); ejes(ax)
plt.suptitle(f'H_x es la tangente en la pose del robot (diferencia con el numérico: {cien(DIF_HX)})', fontsize=11)
plt.tight_layout(); plt.show()

**Qué tiene que salir:** diferencias de 10⁻¹⁰; r = 1,10 m y φ = −24,2°.

## 3. A estima y con el EKF, por la ruta

*En clase · diapositivas 61 y 63*

¿Cuánto se equivoca cada uno, y cómo es su elipse?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# la semilla de los ruidos de la pasada: probad otras, las cifras cambian un poco
SEMILLA = 18  # @param {type:"integer"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
odo, med = simula(SEMILLA)                      # lo que leen la odometría y el sensor en una pasada
Xd, Pd, _, _ = ekf(odo, med, corregir=False)    # a estima
X, P, D2, SALTOS = ekf(odo, med)                # EKF
md_, m = metricas(Xd, Pd), metricas(X, P)
s = np.array([sl for _, sl in SALTOS])          # lo que salta la estimación en cada barrido [m]
semieje = lambda Pk: np.sqrt(np.linalg.eigvalsh(Pk[:2, :2]).max()) * 100   # el semieje mayor de σ [cm]
# para dibujar: el semieje mayor de σ del EKF [cm] (el de la diapositiva 63), la banda de 2,45σ (con el semieje
# mayor) [cm] y las elipses de 2,45σ cada 9 s
SEMIEJE_EKF = np.array([semieje(p) for p in P])
BANDA_D = K95 * np.array([semieje(p) for p in Pd])
BANDA = K95 * SEMIEJE_EKF
ELIPSES_D = [elipse(Pd[k][:2, :2]) + Xd[k, :2] for k in range(90, len(X), 90)]

In [ ]:
# @title ✔ Resultado
print(f'A estima: error medio (RMSE) {coma(md_["rmse"]*100,1)} cm, máximo {coma(md_["maximo"]*100,1)}, final {coma(md_["final"]*100,1)}')
print(f'EKF:      error medio (RMSE) {coma(m["rmse"]*100,1)} cm, máximo {coma(m["maximo"]*100,1)}, final {coma(m["final"]*100,1)}')
print(f'Al corregir, la estimación salta {coma(s.mean()*100,1)} cm de media y {coma(s.max()*100,1)} como mucho')
fig = plt.figure(figsize=(14, 5.2))
gs = fig.add_gridspec(2, 2, width_ratios=[1.35, 1], hspace=0.55, wspace=0.18)
a1 = fig.add_subplot(gs[:, 0])
dibuja_mapa(a1)
a1.plot(VERDAD[:, 0], VERDAD[:, 1], color='black', lw=4.5, alpha=0.25, solid_capstyle='round', label='verdad')
a1.plot(Xd[:, 0], Xd[:, 1], '--', color=ROJO, lw=1.6, label='a estima')
a1.plot(X[:, 0], X[:, 1], color=IQS_AZUL, lw=1.4, label='EKF')
for e in ELIPSES_D:
    a1.plot(e[:, 0], e[:, 1], color=ROJO, lw=1)
a1.plot([], [], color=ROJO, lw=1, label='su elipse de 2,45σ, cada 9 s')
a1.legend(fontsize=9, loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, frameon=False)
for fila, (err, banda, col, t) in enumerate(((md_['err'], BANDA_D, ROJO, 'A estima'), (m['err'], BANDA, IQS_AZUL, 'EKF'))):
    ax = fig.add_subplot(gs[fila, 1])
    ax.fill_between(T_PASOS, 0, banda, color=col, alpha=0.15, lw=0, label='2,45σ (semieje mayor)')
    ax.plot(T_PASOS, err * 100, color=col, lw=1.6, label='error')
    if fila == 1:                                   # el EKF: su σ, como en la diapositiva 63 (en negro, para distinguirla)
        ax.plot(T_PASOS, SEMIEJE_EKF, ':', color='black', lw=1.4, label='σ (semieje mayor)')
    ax.set_ylim(0, (1.1 if fila == 0 else 1.5) * max(banda.max(), 100 * err.max()))   # el EKF: sitio para la leyenda
    ax.set_ylabel('cm'); ax.set_title(f'{t}: el error y su 2,45σ', fontsize=11)
    ax.legend(fontsize=9, loc='upper left', ncol=1 if fila == 0 else 3, columnspacing=0.8, handlelength=1.6, handletextpad=0.4); ejes(ax)
ax.set_xlabel('t (s)')
plt.show()

**Qué tiene que salir:** 27,4 cm de error medio a estima; 4,0 con EKF; figura: elipses rojas crecientes.

## 4. Cómo se juzga un filtro: las métricas

*En clase · diapositiva 65*

¿Cae el error dentro de 2,45σ el 95 % del tiempo?

d² = νᵀ·S⁻¹·ν: de media 2; por encima de 9,21, el 1 % de las buenas.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
d2 = np.array([d[3] for d in D2])                 # la d² de cada medida del EKF
D2_MEDIA = d2.mean()
D2_599, D2_921 = 100*(d2>5.99).mean(), 100*(d2>9.21).mean()   # el % por encima de 5,99 y de 9,21
# para dibujar: la densidad de d² si las medidas son buenas (χ² con dos grados de libertad)
D2_EJE = np.linspace(0, max(12.0, d2.max()), 200)
D2_CHI2 = np.exp(-D2_EJE / 2) / 2

In [ ]:
# @title ✔ Resultado
print(f'{"":10s}{"RMSE":>9s}{"máximo":>9s}{"final":>9s}{"dentro de 2,45σ":>17s}')
for nombre, mm in (('a estima', md_), ('EKF', m)):
    print(f'{nombre:10s}{coma(mm["rmse"]*100,1):>7s} cm{coma(mm["maximo"]*100,1):>7s} cm{coma(mm["final"]*100,1):>7s} cm{coma(mm["dentro"],1):>15s} %')
print(f'd² de las {len(d2)} medidas: media {coma(D2_MEDIA)} ; por encima de 5,99, el {coma(D2_599,1)} % ; de 9,21, el {coma(D2_921,1)} %')
fig, axs = plt.subplots(1, 3, figsize=(14, 3.8), gridspec_kw={'width_ratios': [1.3, 0.8, 1.3]})
for i, (nombre, mm, col) in enumerate((('a estima', md_, ROJO), ('EKF', m, IQS_AZUL))):
    v = [mm['rmse'] * 100, mm['maximo'] * 100, mm['final'] * 100]
    bb = axs[0].bar(np.arange(3) + (i - 0.5) * 0.38, v, 0.36, color=col, label=nombre)
    axs[0].bar_label(bb, labels=[coma(x, 1) for x in v], fontsize=9)
    bb = axs[1].bar(i, mm['dentro'], 0.6, color=col)
    axs[1].bar_label(bb, labels=[coma(mm['dentro'], 1) + ' %'], fontsize=10, label_type='center', color='white', fontweight='bold')
axs[0].set_xticks(range(3)); axs[0].set_xticklabels(['error medio (RMSE)', 'máximo', 'final'])
axs[0].set_ylabel('cm'); axs[0].set_ylim(0, 120 * md_['maximo']); axs[0].legend(fontsize=9); axs[0].set_title('El error', fontsize=11)
axs[1].axhline(95, color='black', ls='--', lw=1.2, label='lo que debe: 95 %'); axs[1].legend(fontsize=9, loc='upper center')
axs[1].set_xticks(range(2)); axs[1].set_xticklabels(['a estima', 'EKF']); axs[1].set_ylim(0, 128); axs[1].set_yticks(range(0, 101, 20))
axs[1].set_ylabel('% de los pasos'); axs[1].set_title('Dentro de su elipse de 2,45σ', fontsize=11)
axs[2].hist(d2, bins=np.arange(0, D2_EJE[-1] + 0.5, 0.5), density=True, color=IQS_AZUL, alpha=0.6, label=f'las {len(d2)} medidas del EKF')
axs[2].plot(D2_EJE, D2_CHI2, color='black', lw=1.6, label='si son buenas: χ², 2 g. l.')
for u, ls, p_ in ((5.99, ':', 5), (9.21, '--', 1)):
    axs[2].axvline(u, color='black', ls=ls, lw=1.2); axs[2].text(u + 0.15, 0.3, f'{coma(u)} ({p_} %)', fontsize=9)
axs[2].set_xlabel('d²'); axs[2].set_ylabel('densidad'); axs[2].set_ylim(0, 0.6)
axs[2].set_title(f'La innovación: media {coma(D2_MEDIA)}', fontsize=11); axs[2].legend(fontsize=9, loc='upper right')
axs[0].yaxis.set_major_formatter(COMA); axs[1].yaxis.set_major_formatter(COMA); ejes(axs[2])
for ax in axs:
    ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

**Qué tiene que salir:** a estima, 100 % dentro; EKF, 95,0 %; d², 1,99 de media.

## 5. Quitar balizas

*En clase · diapositiva 65*

Solo con la nave de abajo: ¿qué pasa en el pasillo?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# las balizas que están en el mapa, por su número (la figura de la sección 1): de la 0 a la 4, las de la nave
# de abajo; las diez, (0, 1, 2, 3, 4, 5, 6, 7, 8, 9)
BALIZAS_EN_MAPA = (0, 1, 2, 3, 4)  # @param {type:"raw"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
Xb, Pb, _, SALb = ekf(odo, med, balizas=list(BALIZAS_EN_MAPA))
mb = metricas(Xb, Pb)
sin = [k for k in med if not any(j in BALIZAS_EN_MAPA for j, _ in med[k])]   # los barridos sin ninguna baliza del mapa
SEMIEJE_MAX_B = max(semieje(p) for p in Pb)                                    # [cm]
K_SALTO_B, SALTO_MAX_B = max(SALb, key=lambda ks: ks[1])                       # el salto más grande: su paso y cuánto [m]
BANDA_B = K95 * np.array([semieje(p) for p in Pb])                             # para dibujar: 2,45σ [cm]
# para dibujar: los tramos de barridos seguidos sin balizas del mapa, (primer paso, último paso); uno suelto, ± medio barrido
TRAMOS_SIN = []
for k_sin in sin:
    if TRAMOS_SIN and k_sin - TRAMOS_SIN[-1][1] == CADA:
        TRAMOS_SIN[-1][1] = k_sin
    else:
        TRAMOS_SIN.append([k_sin, k_sin])
TRAMOS_SIN = [(k0, k1) if k1 > k0 else (k0 - CADA // 2, k1 + CADA // 2) for k0, k1 in TRAMOS_SIN]
FUERA_B = [j for j in range(len(BALIZAS)) if j not in BALIZAS_EN_MAPA]          # las que no están en el mapa

In [ ]:
# @title ✔ Resultado
print(f'Barridos sin ninguna baliza: {len(sin)}' + (f' (de t = {coma(sin[0]/10, 0)} a {coma(sin[-1]/10, 0)} s)' if sin else ''))
print(f'RMSE {coma(mb["rmse"]*100,1)} cm, máximo {coma(mb["maximo"]*100,1)} cm, dentro de 2,45σ el {coma(mb["dentro"],1)} %')
print(f'El semieje mayor llega a {coma(SEMIEJE_MAX_B,1)} cm ; el salto más grande, {coma(SALTO_MAX_B*100,1)} cm')
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.6), gridspec_kw={'width_ratios': [1.15, 1]})
dibuja_mapa(a1, en_mapa=BALIZAS_EN_MAPA, numeros=True)
for i, (k0, k1) in enumerate(TRAMOS_SIN):      # la etiqueta, solo en el primer tramo
    a1.plot(VERDAD[k0:k1 + 1, 0], VERDAD[k0:k1 + 1, 1], color=AMBAR, lw=12, alpha=0.5, solid_capstyle='butt',
            label='_nolegend_' if i else 'sin balizas del mapa a la vista')
    a2.axvspan(k0 / 10, k1 / 10, color=AMBAR, alpha=0.2, lw=0, label='_nolegend_' if i else 'barridos sin balizas del mapa')
a1.plot(VERDAD[:, 0], VERDAD[:, 1], color='black', lw=4.5, alpha=0.25, solid_capstyle='round', label='verdad')
a1.plot(Xb[:, 0], Xb[:, 1], color=IQS_AZUL, lw=1.4, label='EKF')
if FUERA_B:
    a1.plot([], [], '*', mfc='white', mec=GRIS, ms=12, label='fuera del mapa')
a1.legend(fontsize=9, loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, frameon=False)
a2.fill_between(T_PASOS, 0, BANDA_B, color=IQS_AZUL, alpha=0.15, lw=0, label='2,45σ (semieje mayor)')
if FUERA_B:
    a2.plot(T_PASOS, m['err'] * 100, color=GRIS, lw=1, label='con las diez')
a2.plot(T_PASOS, mb['err'] * 100, color=IQS_AZUL, lw=1.6, label='error')
a2.annotate(f'salta {coma(SALTO_MAX_B*100,1)} cm', xy=(K_SALTO_B / 10, mb['err'][K_SALTO_B] * 100), xytext=(K_SALTO_B / 10 + 6, 0.55 * BANDA_B.max()),
            fontsize=10, color=IQS_AZUL, arrowprops=dict(arrowstyle='->', color=IQS_AZUL),
            bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.85))
a2.set_xlabel('t (s)'); a2.set_ylabel('cm'); a2.set_ylim(0, 1.1 * BANDA_B.max()); a2.legend(fontsize=9, loc='upper right')
a2.set_title('El error y su 2,45σ, con las balizas del mapa', fontsize=11); ejes(a2)
plt.tight_layout(); plt.show()

**Qué tiene que salir:** 12 s sin balizas; 12,5 cm de error medio; al volver, salta 15,4 cm.

## 6. La baliza lejana

*En clase · diapositiva 62*

Añadid distancias: ¿desde dónde apenas sitúa de lado?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# las distancias a la baliza [m]: probad 3, 4, 5, 6...
DISTANCIAS = (2.0, 8.0)  # @param {type:"raw"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
P0 = np.diag([0.10**2, 0.10**2, np.radians(2.0)**2])      # lo que cree antes de leer: 10 cm, 10 cm y 2°
LECTURAS = []           # (distancia [m], σ a lo largo [cm], de lado [cm], del rumbo [°], 1° de rumbo de lado [cm], la Σ de después)
for D in DISTANCIAS:
    Hx = jac_h(np.zeros(3), np.array([D, 0.0]))
    S_ = Hx @ P0 @ Hx.T + R
    Lg = P0 @ Hx.T @ np.linalg.inv(S_)
    Pp = (np.eye(3) - Lg @ Hx) @ P0
    LECTURAS.append((D, np.sqrt(Pp[0,0])*100, np.sqrt(Pp[1,1])*100, np.degrees(np.sqrt(Pp[2,2])), D*np.pi/180*100, Pp))
# para dibujar: las elipses de 1σ de antes y de después [cm], el orden de cerca a lejos y un color por orden,
# de la más cerca (verde) a la más lejos (rojo)
ELIPSE_P0 = elipse(P0[:2, :2] * 1e4, 1.0)
ELIPSES_6 = [elipse(Pp[:2, :2] * 1e4, 1.0) for *_, Pp in LECTURAS]
ORDEN_6 = sorted(range(len(LECTURAS)), key=lambda i: LECTURAS[i][0])
from matplotlib.colors import LinearSegmentedColormap
CERCA_LEJOS = LinearSegmentedColormap.from_list('cerca_lejos', [IQS_VERDE, AMBAR, ROJO])
COLOR_6 = [None] * len(LECTURAS)
for r_, i in enumerate(ORDEN_6):
    COLOR_6[i] = CERCA_LEJOS(r_ / max(len(ORDEN_6) - 1, 1))

In [ ]:
# @title ✔ Resultado
for D, sl, sd, sr, grado, _ in LECTURAS:
    print(f'a {coma(D, 1)} m: σ a lo largo {coma(sl,1)} cm, de lado {coma(sd,1)} cm,'
          f' rumbo {coma(sr,2)}° ; 1° de rumbo son {coma(grado,1)} cm de lado')
fig, axs = plt.subplots(1, 3, figsize=(14, 4.3), gridspec_kw={'width_ratios': [1, 1.15, 1.15]})
axs[0].plot(ELIPSE_P0[:, 0], ELIPSE_P0[:, 1], color=GRIS, lw=1.6, label='antes: 10 cm y 2°')
for i in ORDEN_6:                                 # de cerca a lejos
    axs[0].plot(ELIPSES_6[i][:, 0], ELIPSES_6[i][:, 1], color=COLOR_6[i], lw=2, label=f'tras leer a {coma(LECTURAS[i][0], 1)} m')
axs[0].set_aspect('equal'); axs[0].set_xlim(-12, 12); axs[0].set_ylim(-12, 12)
axs[0].set_xlabel('a lo largo (cm)'); axs[0].set_ylabel('de lado (cm)'); axs[0].set_title('La elipse de 1σ', fontsize=11)
fig.legend(*axs[0].get_legend_handles_labels(), fontsize=9, loc='lower center', ncol=min(len(LECTURAS) + 1, 7))
for ax, i, antes, umbral, u, t in ((axs[1], 2, 10, 9, 'σ de lado (cm)', 'De lado'), (axs[2], 3, 2, None, 'σ del rumbo (°)', 'El rumbo')):
    ax.scatter([L[0] for L in LECTURAS], [L[i] for L in LECTURAS], c=COLOR_6, s=50, zorder=2)
    ax.axhline(antes, color=GRIS, lw=1.2, label='antes de leer')
    if umbral:
        ax.axhline(umbral, color='black', ls='--', lw=1.2, label='9 cm: apenas sitúa')
    ax.set_xlim(0, 1.1 * max(9, max(DISTANCIAS))); ax.set_ylim(0, 1.15 * antes)
    ax.set_xlabel('distancia a la baliza (m)'); ax.set_ylabel(u); ax.set_title(t + ', tras una lectura', fontsize=11)
    ax.legend(fontsize=9, loc='lower right'); ejes(ax)
ejes(axs[0]); plt.tight_layout(rect=(0, 0.08, 1, 1)); plt.show()

**Qué tiene que salir:** de lado, 6,2 cm a 2 m y 9,5 a 8 m; el rumbo, 1,67° y 1,05°; figura: de lejos, se estira de lado.

## 7. Una asociación equivocada, y la puerta de Mahalanobis

*En clase · diapositiva 65*

La 9, tomada por la 4: ¿cuánto salta, y qué dice d²?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# la medida equivocada: (barrido, baliza de verdad, baliza a la que se atribuye); el barrido 200 es t = 20 s
CAMBIO = (200, 9, 4)  # @param {type:"raw"}
# la puerta de Mahalanobis: se descartan las medidas con d² por encima (9,21 deja fuera el 1 % de las buenas)
PUERTA = 9.21  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
CASOS_7 = []            # (puerta, X, métricas, d² de la equivocada, t de cada medida [s], d² de cada medida)
for puerta in (None, PUERTA):
    Xw, Pw, D2w, _ = ekf(odo, med, cambia=CAMBIO, puerta=puerta)
    mw = metricas(Xw, Pw)
    d2w = [d[3] for d in D2w if d[0] == CAMBIO[0] and d[1] == CAMBIO[1]][0]
    CASOS_7.append((puerta, Xw, mw, d2w, np.array([d[0] for d in D2w]) / 10, np.array([d[3] for d in D2w])))

In [ ]:
# @title ✔ Resultado
for puerta, Xw, mw, d2w, _, _ in CASOS_7:
    etiqueta = 'sin puerta' if puerta is None else f'con la puerta en {coma(puerta)}'
    print(f'{etiqueta}: d² de la equivocada = {coma(d2w,0)} ; error justo después {coma(mw["err"][CAMBIO[0]]*100,1)} cm ;'
          f' RMSE {coma(mw["rmse"]*100,1)} cm ; dentro de 2,45σ el {coma(mw["dentro"],1)} %')
fig, (a1, a2) = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
a1.plot(T_PASOS, m['err'] * 100, color=GRIS, lw=1, label='sin equivocarse')
for (puerta, Xw, mw, d2w, td, dd), col, ls, mk in zip(CASOS_7, (ROJO, IQS_AZUL), ('-', '--'), ('o', 'x')):
    t_ = 'sin puerta' if puerta is None else f'con la puerta en {coma(puerta)}'
    a1.plot(T_PASOS, mw['err'] * 100, ls, color=col, lw=1.6, label=t_)
    a2.plot(td, dd, mk, color=col, ms=4, alpha=0.8, label=t_)
a1.axvline(CAMBIO[0] / 10, color=AMBAR, lw=1.2, ls=':')
a1.text(CAMBIO[0] / 10 + 2.5, 0.92 * CASOS_7[0][2]['err'].max() * 100, f'la equivocada: {coma(CASOS_7[0][2]["err"][CAMBIO[0]]*100,1)} cm',
        fontsize=10, color=ROJO, va='top')
a1.set_ylabel('error (cm)'); a1.legend(fontsize=9, loc='upper right'); a1.set_title('El error de posición', fontsize=11); ejes(a1)
a2.axhline(PUERTA, color='black', ls='--', lw=1.2, label=f'la puerta: {coma(PUERTA)}')
a2.plot(CAMBIO[0] / 10, CASOS_7[0][3], '*', color=AMBAR, ms=16, mec='black', zorder=5, label=f'la equivocada: {coma(CASOS_7[0][3], 0)}')
a2.set_yscale('log'); a2.set_ylim(1e-3, 3 * max(CASOS_7[0][3], PUERTA)); a2.yaxis.set_minor_formatter(plt.NullFormatter())
a2.set_xlabel('t (s)'); a2.set_ylabel('d² (escala log.)'); a2.legend(fontsize=9, loc='lower right', ncol=2)
a2.set_title('La sorpresa de cada medida, d²', fontsize=11); ejes(a2)
plt.tight_layout(); plt.show()

**Qué tiene que salir:** d² = 2580 y un salto a 118,2 cm; con la puerta, se descarta.

### ✎ Vuestra respuesta (tres líneas)

*Doble clic aquí y escribid, con cifras. Al acabar la clase, Archivo → Descargar → Descargar .ipynb de los dos cuadernos, el S18a y el S18b, y a la Tarea «Taller S18: estimación y localización»: cada uno, los suyos.*

1. A estima y con EKF: error medio y % dentro de 2,45σ; ¿es honesta la elipse?
2. Sin las balizas de arriba, el pasillo; o la lejana: desde dónde apenas sitúa de lado.
3. La equivocada: cuánto salta, qué dice d² y qué hace la puerta.

## 8. Romper el EKF (en casa)

*En casa · diapositiva 60*

El rumbo de salida, 90° mal, y una sola baliza: ¿avisa?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# el error del rumbo al salir del muelle [°]
ERROR_RUMBO = 90.0  # @param {type:"number"}
# lo que cree el filtro de ese rumbo: su σ al salir [°] (probad 1: se cree seguro)
SIGMA_RUMBO_SALIDA = 90.0  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
x0 = VERDAD[0] + np.array([0, 0, np.radians(ERROR_RUMBO)])
S0 = np.diag([0.02**2, 0.02**2, np.radians(SIGMA_RUMBO_SALIDA)**2])
CASOS_8 = []            # (nombre, X, d², métricas, σ de posición al final [m], para dibujar: 2,45σ [m])
for nombre, bal in (('una baliza', [1]), ('diez balizas', range(10))):
    Xr, Pr, D2r, _ = ekf(odo, med, balizas=list(bal), x0=x0, Sigma0=S0)
    mr = metricas(Xr, Pr)
    sf = np.sqrt(Pr[-1, 0, 0] + Pr[-1, 1, 1])
    CASOS_8.append((nombre, Xr, D2r, mr, sf, K95 * np.array([semieje(p) for p in Pr]) / 100))

In [ ]:
# @title ✔ Resultado
for nombre, Xr, D2r, mr, sf, _ in CASOS_8:
    print(f'{nombre}: error máximo {coma(mr["maximo"],2)} m, final {coma(mr["final"]*100,1)} cm con σ = {coma(sf*100,1)} cm'
          f' ({coma(mr["final"]/sf,1)}σ) ; últimas d²: {", ".join(coma(d[3],1) for d in D2r[-5:])}')
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.6), gridspec_kw={'width_ratios': [1.15, 1]})
dibuja_mapa(a1, en_mapa=[1], numeros=True)
a1.plot(VERDAD[:, 0], VERDAD[:, 1], color='black', lw=4.5, alpha=0.25, solid_capstyle='round', label='verdad')
for (nombre, Xr, D2r, mr, sf, b245), col in zip(CASOS_8, (ROJO, IQS_AZUL)):
    a1.plot(Xr[:, 0], Xr[:, 1], color=col, lw=1.4, label=f'EKF, {nombre}')
    a2.plot(T_PASOS, mr['err'], color=col, lw=1.8, label=f'error, {nombre}')
    a2.plot(T_PASOS, b245, '--', color=col, lw=1.2, label=f'lo que cree (2,45σ), {nombre}')
a1.plot([], [], '*', color=IQS_VERDE, ms=12, label='la 1: la única del de una')
a1.plot([], [], '*', mfc='white', mec=GRIS, ms=12, label='las otras: solo en el de diez')
a1.legend(fontsize=9, loc='lower left', bbox_to_anchor=(0, 1.0), ncol=3, frameon=False)
a1.set_xlim(-2.5, 13.7); a1.set_ylim(-2.5, 9.7)
a2.set_yscale('log'); a2.set_ylim(0.005, 80); a2.yaxis.set_minor_formatter(plt.NullFormatter())
a2.set_xlabel('t (s)'); a2.set_ylabel('m (escala log.)'); a2.legend(fontsize=9, loc='upper right', ncol=2)
a2.set_title('El error y lo que el filtro cree', fontsize=11); ejes(a2)
plt.tight_layout(); plt.show()

**Qué tiene que salir:** con una baliza, 82,0 cm al final con σ = 12,7 cm (6,4σ).

### Ejercicio 1 (en casa): un sensor de rumbo peor

Con 5°: ¿qué empeora más, posición o rumbo? ¿Sigue honesta la elipse?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# la σ del rumbo del sensor [°]
E1_SIGMA_RUMBO = 5.0  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
e1_antes = (SIG_PHI, R)                       # el sensor de la sección 1: simula y ekf usan SIG_PHI y R
SIG_PHI = np.radians(E1_SIGMA_RUMBO); R = np.diag([SIG_R**2, SIG_PHI**2])
try:
    e1_odo, e1_med = simula(SEMILLA)            # la misma pasada, con el sensor peor
    E1_X, E1_P, _, _ = ekf(e1_odo, e1_med)
finally:
    SIG_PHI, R = e1_antes                       # y se deja como estaba
E1_M = metricas(E1_X, E1_P)
E1_RUMBO = np.degrees(np.sqrt(E1_P[1:, 2, 2]))  # σ del rumbo en cada paso [°]
E1_RUMBO_ANTES = np.degrees(np.sqrt(P[1:, 2, 2]))   # con el sensor de la sección 1

In [ ]:
# @title ✔ Resultado
print(f'Con {coma(E1_SIGMA_RUMBO, 0)}°: RMSE {coma(E1_M["rmse"]*100,1)} cm y máximo {coma(E1_M["maximo"]*100,1)} (con {coma(SIGMA_RUMBO, 0)}°: {coma(m["rmse"]*100,1)} y {coma(m["maximo"]*100,1)}) ;'
      f' σ del rumbo, de media {coma(E1_RUMBO.mean(),2)}° ({coma(E1_RUMBO_ANTES.mean(),2)}°) ; dentro de 2,45σ el {coma(E1_M["dentro"],1)} %')
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 3.6))
for err, rumbo, col, t in ((m['err'], E1_RUMBO_ANTES, IQS_AZUL, f'rumbo del sensor, {coma(SIGMA_RUMBO, 0)}°'),
                           (E1_M['err'], E1_RUMBO, AMBAR, f'rumbo del sensor, {coma(E1_SIGMA_RUMBO, 0)}°')):
    a1.plot(T_PASOS, err * 100, color=col, lw=1.5, label=t)
    a2.plot(T_PASOS[1:], rumbo, color=col, lw=1.5, label=t)
a1.set_xlabel('t (s)'); a1.set_ylabel('cm'); a1.set_title('El error de posición', fontsize=11); a1.legend(fontsize=9, loc='upper left')
a2.set_xlabel('t (s)'); a2.set_ylabel('°'); a2.set_title('La σ del rumbo', fontsize=11); a2.legend(fontsize=9, loc='upper left')
ejes(a1, a2); plt.tight_layout(); plt.show()

### Ejercicio 2 (en casa): ¿cuántas balizas hacen falta?

Quitándolas una a una: ¿con cuántas sigue bajo 5 cm?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# el error medio (RMSE) que se pide [cm]
E2_LIMITE = 5.0  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
e2_rmse = lambda e2_X: np.sqrt((np.linalg.norm(e2_X[:, :2] - VERDAD[:, :2], axis=1)**2).mean())   # el RMSE de metricas [m]
E2 = []                 # (cuántas quedan, cuáles, RMSE [m])
E2_ORDEN = {}           # para dibujar: baliza -> cuántas quedaban cuando se quitó (la que queda al final, 1)
e2_quedan = list(range(10))
while len(e2_quedan) > 1:
    e2_prueba = []
    for e2_j in e2_quedan:
        e2_X, _, _, _ = ekf(odo, med, balizas=[i for i in e2_quedan if i != e2_j])
        e2_prueba.append((e2_rmse(e2_X), e2_j))
    e2_rm, e2_j = min(e2_prueba); E2_ORDEN[e2_j] = len(e2_quedan); e2_quedan.remove(e2_j)
    E2.append((len(e2_quedan), list(e2_quedan), e2_rm))
E2_ORDEN[e2_quedan[0]] = 1
E2_N = [10] + [n for n, _, _ in E2]                          # para dibujar: cuántas quedan y el RMSE [cm]
E2_RMSE_CM = [m['rmse'] * 100] + [rm * 100 for _, _, rm in E2]
e2_bastan = [n for n, _, rm in [(10, None, m['rmse'])] + E2 if rm * 100 < E2_LIMITE]
E2_BASTAN = min(e2_bastan) if e2_bastan else None    # las menos que bastan (None: ni con las diez)
# para dibujar: en qué orden se quitó cada baliza (1.ª, la primera; «queda», la última que queda)
E2_RETIRADA = {e2_j: (f'{len(BALIZAS) + 1 - o}.ª' if o > 1 else 'queda') for e2_j, o in E2_ORDEN.items()}

In [ ]:
# @title ✔ Resultado
for n, cuales, rm in E2:
    print(n, 'balizas:' if n > 1 else 'baliza:', ' '.join(f'({coma(BALIZAS[i][0],1)}; {coma(BALIZAS[i][1],2)})' for i in cuales), '· RMSE', coma(rm*100,1), 'cm')
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.4), gridspec_kw={'width_ratios': [1, 1.15]})
a1.plot(E2_N, E2_RMSE_CM, 'o-', color=IQS_AZUL, lw=1.8)
a1.axhline(E2_LIMITE, color='black', ls='--', lw=1.2, label=f'{coma(E2_LIMITE, 0)} cm')
if E2_BASTAN:
    a1.axvline(E2_BASTAN, color=IQS_VERDE, lw=1.2, label=f'bastan {E2_BASTAN}')
a1.set_xlim(10.5, 0.5); a1.set_xticks(range(1, 11)); a1.set_xlabel('balizas en el mapa'); a1.set_ylabel('error medio (cm)')
a1.set_title('Quitando cada vez la que menos empeora', fontsize=11); a1.legend(fontsize=9, loc='upper left'); ejes(a1)
e2_verdes = [j for j, o in E2_ORDEN.items() if E2_BASTAN and o <= E2_BASTAN]
dibuja_mapa(a2, en_mapa=e2_verdes)
for j, (bx, by) in enumerate(BALIZAS):          # hacia dentro de la nave; las del muro, fuera del muro
    dy = (-0.45 if by < 4.45 else 0.45) if 4 < by < 5 else (0.45 if by < 4.5 else -0.45)
    a2.text(bx, by + dy, E2_RETIRADA[j], fontsize=10, fontweight='bold', va='center', ha='left' if bx < 1 else 'right' if bx > 12 else 'center',
            color=IQS_VERDE if j in e2_verdes else GRIS, bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.85))
a2.set_title('Orden en que se quitan; ' + (f'en verde, las {E2_BASTAN} que bastan' if E2_BASTAN else 'ni con las diez basta'), fontsize=11)
plt.tight_layout(); plt.show()

### Ejercicio 3 (en casa): el umbral de la puerta

Con 9,21, 5,99 y 3: ¿cuántas buenas descarta? ¿Y el error?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# los umbrales de la puerta que se prueban (d²)
E3_UMBRALES = (9.21, 5.99, 3.0)  # @param {type:"raw"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
E3 = []                 # (umbral, medidas descartadas, medidas, métricas)
for e3_u in E3_UMBRALES:
    e3_X, e3_P, e3_D2, _ = ekf(odo, med, puerta=e3_u)
    E3.append((e3_u, sum(d[3] > e3_u for d in e3_D2), len(e3_D2), metricas(e3_X, e3_P)))

In [ ]:
# @title ✔ Resultado
for u, desc, n, mg in E3:
    print('umbral', coma(u), ':', desc, 'de', n, 'descartadas ; RMSE', coma(mg['rmse']*100,1), 'cm ; dentro', coma(mg['dentro'],1), '%')
fig, axs = plt.subplots(1, 3, figsize=(13, 3.4))
x_ = np.arange(len(E3)); et = [coma(u) for u, *_ in E3]
for ax, v, u, t, f in ((axs[0], [d for _, d, _, _ in E3], 'medidas', 'Medidas buenas descartadas', lambda x: str(int(x))),
                       (axs[1], [mg['rmse'] * 100 for *_, mg in E3], 'cm', 'Error medio (RMSE)', lambda x: coma(x, 1)),
                       (axs[2], [mg['dentro'] for *_, mg in E3], '%', 'Dentro de 2,45σ', lambda x: coma(x, 1))):
    bb = ax.bar(x_, v, 0.6, color=IQS_AZUL)
    ax.bar_label(bb, labels=[f(x) for x in v], fontsize=10, **(dict(label_type='center', color='white', fontweight='bold') if u == '%' else {}))
    ax.set_xticks(x_); ax.set_xticklabels(et); ax.set_xlabel('umbral de la puerta (d²)'); ax.set_ylabel(u)
    ax.set_title(t, fontsize=11); ax.set_ylim(0, 1.18 * max(max(v), 1)); ax.yaxis.set_major_formatter(COMA); ax.set_axisbelow(True)
axs[2].axhline(95, color='black', ls='--', lw=1.2); axs[2].text(len(E3) - 0.6, 97, '95 %', fontsize=9, ha='right', va='bottom'); axs[2].set_ylim(0, 112)
plt.tight_layout(); plt.show()

---

## Soluciones

**Ejercicio 1.** Con 5°, el error medio pasa de 4,0 a 5,1 cm y el máximo, de 12,7 a 12,8: la posición la sujetan las distancias, que siguen con 5 cm. El rumbo sí: su σ media, de 0,63° a 1,15°. La elipse sigue honesta (93,2 % dentro) porque R dice la verdad del sensor; si cambiara el sensor y no R, el filtro se creería mejor de lo que es.

**Ejercicio 2.** Con 9, 8 y 7 balizas, 3,8-3,9 cm; con 6, 4,0; con 5, 4,2; con 4 —(0,1; 2,0), (3,0; 0,1), (10,5; 8,9) y (7,0; 4,75)—, 4,5, las menos por debajo de 5 cm; con 3, 5,3; con 2, 6,8; con una, 15,8. Quedan las que se ven desde más tramos, en direcciones distintas: importa que siempre haya alguna a la vista.

**Ejercicio 3.** Con 9,21 se descarta 1 medida buena de 182 (4,0 cm, 95,0 % dentro); con 5,99, 11 (4,3 cm, 93,4 %); con 3, 53 (6,6 cm, 80,3 %). Un umbral bajo tira medidas buenas: el filtro corrige menos y deja de ser honesto. La puerta es para lo imposible, como la asociación equivocada (d² = 2580).

---

## Para llevarse de esta sesión

- **El EKF**: de 27 cm a estima a 4.
- **Se juzga** por el error, su elipse y d².
- **Falla** sin balizas o con una asociación equivocada.

*Cuaderno del curso 82514 Mecatrónica y Robótica · IQS Universitat Ramon Llull · curso 2026/27*

*© Guillermo Reyes Carmenaty · Publicado bajo [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es): puedes usarlo, adaptarlo y redistribuirlo, incluso con fines comerciales, siempre que reconozcas la autoría.*